## GenX test

In [41]:
# include("GenX.jl")
# using .GenX
# using HiGHS
# using JuMP
# using Gurobi
# using DataFrames
# using YAML
# # Genx case_runners file,
# case = "..\\example_systems\\18_markets_co2_test"
# settings_path = joinpath(case, "settings")
# policies_path = joinpath(case, "policies")
# output_folder = joinpath(case, "Results") # Write-output settings YAML file path,
# genx_settings = joinpath(settings_path, "genx_settings.yml") # Settings YAML file path,
# mysetup = GenX.configure_settings(genx_settings, output_folder) # mysetup dictionary stores settings and GenX-specific parameters,
# optimizer = Gurobi.Optimizer
# OPTIMIZER =  GenX.configure_solver(settings_path, optimizer)
# myinputs = GenX.load_inputs(mysetup_local, case)

## others

In [1]:

using JuMP # used for mathematical programming
using DataFrames #This package allows put together data into a matrix
using CSV
using StatsBase
using LinearAlgebra
using YAML
using Dates
using Clustering
using Distances
using Combinatorics
using Random
using RecursiveArrayTools
using Statistics
using HiGHS
using Logging
using Gurobi

using PrecompileTools: @compile_workload

# Global scaling factor used when ParameterScale is on to shift values from MW to GW
# DO NOT CHANGE THIS (Unless you do so very carefully)
# To translate MW to GW, divide by ModelScalingFactor
# To translate $ to $M, multiply by ModelScalingFactor^2
# To translate $/MWh to $M/GWh, multiply by ModelScalingFactor
const ModelScalingFactor = 1e+3

"""
An abstract type that should be subtyped for users creating GenX resources.
"""
abstract type AbstractResource end

# thanks, ChatGPT
function include_all_in_folder(folder)
    base_path = joinpath(@__DIR__, folder)
    for (root, dirs, files) in Base.Filesystem.walkdir(base_path)
        for file in files
            if endswith(file, ".jl")
                include(joinpath(root, file))
            end
        end
    end
end

include_all_in_folder("case_runners")
include_all_in_folder("configure_settings")
include_all_in_folder("configure_solver")
include_all_in_folder("load_inputs")
include_all_in_folder("model")
include_all_in_folder("write_outputs")

include_all_in_folder("multi_stage")
include_all_in_folder("additional_tools")

include("startup/genx_startup.jl")

┌ Info: Running precompile script for GenX. This may take a few minutes.
└ @ Main c:\Users\mjarada1\Desktop\GenX.jl\src\startup\genx_startup.jl:56


In [ ]:
case = "..\\example_systems\\18_markets_co2_test"
genx_settings = get_settings_path(case, "genx_settings.yml") # Settings YAML file path
writeoutput_settings = get_settings_path(case, "output_settings.yml") # Write-output settings YAML file path
mysetup = configure_settings(genx_settings, writeoutput_settings) # mysetup dictionary stores settings and GenX-specific parameters


settings_path = get_settings_path(case)
optimizer = Gurobi.Optimizer
### Configure solver
println("Configuring Solver")
OPTIMIZER = configure_solver(settings_path, optimizer)

#### Running a case

### Load inputs
println("Loading Inputs")
myinputs = load_inputs(mysetup, case)


In [ ]:
inputs = myinputs
setup = mysetup
T = inputs["T"]     # Number of time steps (hours)
Z = inputs["Z"]     # Number of zones

## Start pre-solve timer
presolver_start_time = time()

# Generate Energy Portfolio (EP) Model
EP = Model(OPTIMIZER)
set_string_names_on_creation(EP, Bool(setup["EnableJuMPStringNames"]))

# Introduce dummy variable fixed to zero to ensure that expressions like eTotalCap,
# eTotalCapCharge, eTotalCapEnergy and eAvail_Trans_Cap all have a JuMP variable
@variable(EP, vZERO==0)

# Initialize Power Balance Expression
# Expression for "baseline" power balance constraint
create_empty_expression!(EP, :ePowerBalance, (T, Z))

# Initialize Objective Function Expression
EP[:eObj] = AffExpr(0.0)

create_empty_expression!(EP, :eGenerationByZone, (Z, T))

# Energy losses related to technologies
create_empty_expression!(EP, :eELOSSByZone, Z)

# Total Generation from all resources
create_empty_expression!(EP, :eTotalGenerationByZone, (Z, T))

# Curtailment from all resources
# if resource is a renewable resource it is equal MaxCapacity-vP, otherwise = 0
G = inputs["G"]
create_empty_expression!(EP, :eCurtailment, (G, T))

# Initialize Capacity Reserve Margin Expression
if setup["CapacityReserveMargin"] > 0
    create_empty_expression!(EP,
        :eCapResMarBalance,
        (inputs["NCapacityReserveMargin"], T))
end

# Energy Share Requirement
if setup["EnergyShareRequirement"] >= 1
    create_empty_expression!(EP, :eESR, inputs["nESR"])
end

# Hourly Matching Requirement
if setup["HourlyMatching"] == 1
    create_empty_expression!(EP, :eHM, (T, Z))
end

if setup["MinCapReq"] == 1
    create_empty_expression!(EP, :eMinCapRes, inputs["NumberOfMinCapReqs"])
end

if setup["MaxCapReq"] == 1
    create_empty_expression!(EP, :eMaxCapRes, inputs["NumberOfMaxCapReqs"])
end

if setup["HydrogenMinimumProduction"] > 0
    create_empty_expression!(EP, :eH2DemandRes, inputs["NumberOfH2DemandReqs"])
end



# Infrastructure
discharge!(EP, inputs, setup)

non_served_energy!(EP, inputs, setup)

investment_discharge!(EP, inputs, setup)

if setup["UCommit"] > 0
    ucommit!(EP, inputs, setup)
end

fuel!(EP, inputs, setup)

co2!(EP, inputs)

if setup["OperationalReserves"] > 0
    operational_reserves!(EP, inputs, setup)
end

if Z > 1
    investment_transmission!(EP, inputs, setup)
    transmission!(EP, inputs, setup)
end

if Z > 1 && setup["DC_OPF"] != 0
    dcopf_transmission!(EP, inputs, setup)
end

# Technologies
# Model constraints, variables, expression related to dispatchable renewable resources

if !isempty(inputs["VRE"])
    curtailable_variable_renewable!(EP, inputs, setup)
end

# Model constraints, variables, expression related to non-dispatchable renewable resources
if !isempty(inputs["MUST_RUN"])
    must_run!(EP, inputs, setup)
end

# Model constraints, variables, expression related to energy storage modeling
if !isempty(inputs["STOR_ALL"])
    storage!(EP, inputs, setup)
end

# Model constraints, variables, expression related to reservoir hydropower resources
if !isempty(inputs["HYDRO_RES"])
    hydro_res!(EP, inputs, setup)
end

# Model constraints, variables, expression related to reservoir hydropower resources with long duration storage
if inputs["REP_PERIOD"] > 1 && !isempty(inputs["STOR_HYDRO_LONG_DURATION"])
    hydro_inter_period_linkage!(EP, inputs)
end

# Model constraints, variables, expression related to demand flexibility resources
if !isempty(inputs["FLEX"])
    flexible_demand!(EP, inputs, setup)
end

# Model constraints, variables, expression related to thermal resource technologies
if !isempty(inputs["THERM_ALL"])
    thermal!(EP, inputs, setup)
end

# Model constraints, variables, expression related to retrofit technologies
if !isempty(inputs["RETROFIT_OPTIONS"])
    EP = retrofit(EP, inputs)
end

# Model constraints, variables, expressions related to the co-located VRE-storage resources
# if !isempty(inputs["VRE_STOR"])
#     vre_stor!(EP, inputs, setup)
# end

# Model constraints, variables, expressions related to telectrolyzers
if !isempty(inputs["ELECTROLYZER"]) ||
   (!isempty(inputs["VRE_STOR"]) && !isempty(inputs["VS_ELEC"]))
    electrolyzer!(EP, inputs, setup)
end

In [45]:
G = inputs["G"]     # Number of generators
T = inputs["T"]     # Number of time steps
MZ = inputs["MZ"]     # Number of markets
Z = inputs["Z"]
# generators_markets = inputs["Generator_Market"]

### Variables ###
# 1- Amount of energy purchased/imported by each zone z at time t from the associated market
@variable(EP, vMBUY[m in MZ, t = 1:T] >= 0);   
@variable(EP, vMSELL[m in MZ, t = 1:T] >= 0);

# # 3- Amount of energy sold/exported by each zone z at time t to the associated market   
LZ = [k for (k, v) in inputs["LZ_Markets"] if !isempty(v)]
@constraint(EP, cMaxSell[z in LZ, t = 1:T], EP[:eTotalGenerationByZone][z,t] >= sum(vMSELL[m,t] for m in inputs["LZ_Markets"][z] ) )

### Constraints ###
# 1. Maximum energy to buy from market or sell to market
@constraint(EP, cMaxMarketBuy[m in MZ, t = 1:T], vMBUY[m, t] <= inputs["Mrkt_Max_Buy"][m] ) 
@constraint(EP, cMaxMarketSell_1[m in MZ, t = 1:T], vMSELL[m, t] <= inputs["Mrkt_Max_Sell"][m] )

# 4. Power balance constraint       
@expression(EP, eZonalNetMarkets[t = 1:T, z =1:Z], 
if z in MZ
    vMBUY[z,t] - vMSELL[z,t] 
else
    EP[:vZERO]
end)

# Add market purchased and sold energy to power balance expression
add_similar_to_expression!(EP[:ePowerBalance], eZonalNetMarkets)

In [50]:
unregister(EP, :eMPurshaseEmissions)

In [ ]:
@expression(EP, eMPurshaseEmissions[m in 1:Z, t = 1:T], 
if m in MZ 
    vMBUY[m,t] * inputs["Mrkt_CO2_tons_MWh"][m] 
else
    EP[:vZERO]
end)

In [ ]:
typeof(eMPurshaseEmissions[1,1])

In [63]:
create_empty_expression!(EP, :eTotalEmissionsByZone, (Z, T))

In [68]:
for m in 1:Z, t = 1:T
    #add_to_expression!(EP[:eTotalEmissionsByZone][m,t], eMPurshaseEmissions[m,t])
    add_to_expression!(EP[:eTotalEmissionsByZone][m,t], EP[:eEmissionsByZone][m,t])
end

In [ ]:
sum(EP[:eEmissionsByPlant][y, 1] for y in resources_in_zone_by_rid(gen, 2))

In [ ]:
gen = inputs["RESOURCES"]
for z in 1:Z, t = 1:T
    add_term_to_expression!(EP[:eEmissionsByZone][z,t], 
    sum(EP[:eEmissionsByPlant][y, t] for y in resources_in_zone_by_rid(gen, z)))
end

In [ ]:
EP[:eObj]

In [ ]:
@expression(EP, tst[m in 1:Z, t = 1:T], EP[:eEmissionsByZone][m,t] + EP[:eMPurshaseEmissions][m,t])

In [ ]:

if Z > 1
    investment_transmission!(EP, inputs, setup)
    transmission!(EP, inputs, setup)
end

if Z > 1 && setup["DC_OPF"] != 0
    dcopf_transmission!(EP, inputs, setup)
end

# Technologies
# Model constraints, variables, expression related to dispatchable renewable resources

if !isempty(inputs["VRE"])
    curtailable_variable_renewable!(EP, inputs, setup)
end

# Model constraints, variables, expression related to non-dispatchable renewable resources
if !isempty(inputs["MUST_RUN"])
    must_run!(EP, inputs, setup)
end

# Model constraints, variables, expression related to energy storage modeling
if !isempty(inputs["STOR_ALL"])
    storage!(EP, inputs, setup)
end

# Model constraints, variables, expression related to reservoir hydropower resources
if !isempty(inputs["HYDRO_RES"])
    hydro_res!(EP, inputs, setup)
end

# Model constraints, variables, expression related to reservoir hydropower resources with long duration storage
if inputs["REP_PERIOD"] > 1 && !isempty(inputs["STOR_HYDRO_LONG_DURATION"])
    hydro_inter_period_linkage!(EP, inputs)
end

# Model constraints, variables, expression related to demand flexibility resources
if !isempty(inputs["FLEX"])
    flexible_demand!(EP, inputs, setup)
end

# Model constraints, variables, expression related to thermal resource technologies
if !isempty(inputs["THERM_ALL"])
    thermal!(EP, inputs, setup)
end

# Model constraints, variables, expression related to retrofit technologies
if !isempty(inputs["RETROFIT_OPTIONS"])
    EP = retrofit(EP, inputs)
end

# Model constraints, variables, expressions related to the co-located VRE-storage resources
if !isempty(inputs["VRE_STOR"])
    vre_stor!(EP, inputs, setup)
end

# Model constraints, variables, expressions related to telectrolyzers
if !isempty(inputs["ELECTROLYZER"]) ||
   (!isempty(inputs["VRE_STOR"]) && !isempty(inputs["VS_ELEC"]))
    electrolyzer!(EP, inputs, setup)
end
# Policies

if setup["OperationalReserves"] > 0
    operational_reserves_constraints!(EP, inputs)
end